# 사전 등록 검증: 출생 월간 경보로 통일 규칙 5차 vs 6차

- **사전 등록 문서:** [`preregistration_births_v5_v6.md`](preregistration_births_v5_v6.md) — 커밋 `07cff83`. 출생 값을 받기 **전**에 작성하고, 6차 코드(`kosis_monitor.py`의 `v6`)와 `TOPICS["출생"]` 설정도 함께 커밋.
- **자료:** 통계청 KOSIS `DT_1B81A01` 시군구 월별 출생 (1997.01~2025.12), [`data_kosis/pipeline_births.csv`](data_kosis/pipeline_births.csv).
- **보정** 2003~2012, **채점** 2013~2025. 평가 대상: 직전 연도 출생 300명 이상.
- **통과:** 전체 잘못된 경보 ≤ 10% 그리고 모든 규모 구간 ≤ 10%.
- **결정:** 6차만 통과 → 6차 / 둘 다 통과 → 6차 감지율이 5차 + 2%p 이상일 때만 6차 / 그 밖 → 5차 유지.

코랩에서 그대로 실행됩니다 (저장소에서 코드·자료를 받아 옴, API 키 불필요).

In [1]:
import os, urllib.request, warnings; warnings.filterwarnings("ignore")
RAW = "https://raw.githubusercontent.com/quanter02/chelsea-analytics/claude/great-dirac-3wwgzy/experiments/"
if not os.path.exists("kosis_monitor.py"): urllib.request.urlretrieve(RAW + "kosis_monitor.py", "kosis_monitor.py")
import pandas as pd, kosis_monitor as K
tidy = K.fetch(K.TOPICS["출생"], verbose=False)     # 저장본 → 저장소 → KOSIS 순서
print(len(tidy), "행,", tidy.ym.min(), "~", tidy.ym.max())

96492 행, 199701 ~ 202512


## 1. 공식 채점 (채점 기간 2013~2025)

In [2]:
tp = K.TOPICS["출생"]
out = {r: K.run(tp, tidy=tidy, rule=r, verbose=False) for r in ["v5", "v6"]}
def passed(o): return o["test"]["잘못된경보율"] <= 0.10 and (o["groups"]["잘못된경보율"] <= 0.10).all()
rows = []
for r, label in [("v5", "5차 (현재 채택)"), ("v6", "6차 (후보)")]:
    o, cb = out[r], out[r]["calibration"]
    rows.append(dict(규칙=label, α=cb.alpha, m=round(cb.m, 3), 감지율=f"{o['test']['감지율']:.1%}",
                     상반기내=f"{o['test']['상반기내_감지율']:.1%}", 잘못된경보=f"{o['test']['잘못된경보율']:.1%}",
                     구간최대_잘못된경보=f"{o['groups']['잘못된경보율'].max():.1%}", 통과=passed(o)))
pd.DataFrame(rows)

,규칙,α,m,감지율,상반기내,잘못된경보,구간최대_잘못된경보,통과
0,5차 (현재 채택),0.99,0.087,88.1%,53.2%,1.9%,3.0%,True
1,6차 (후보),0.99,0.058,93.5%,58.6%,2.9%,6.9%,True


In [3]:
p5, p6 = passed(out["v5"]), passed(out["v6"])
d5, d6 = out["v5"]["test"]["감지율"], out["v6"]["test"]["감지율"]
if p6 and not p5: verdict = "6차 채택 (6차만 통과)"
elif p5 and p6: verdict = "6차 채택" if d6 >= d5 + 0.02 else "5차 유지 (차이 2%p 미만)"
else: verdict = "5차 유지" + (" (둘 다 실패: 출생에서는 미검증)" if not p5 else "")
print(f"감지율 5차 {d5:.1%} → 6차 {d6:.1%} (차이 {(d6 - d5) * 100:+.1f}%p)")
print("결정:", verdict)

감지율 5차 88.1% → 6차 93.5% (차이 +5.4%p)
결정: 6차 채택


## 2. 규모 구간별

In [4]:
for r in ["v5", "v6"]:
    print(r); display(out[r]["groups"])

v5


,단위연도,이탈해,감지율,정상해,잘못된경보율
규모,,,,,
300~1천,771.0,280.0,0.707,285.0,0.004
1천~3천,904.0,293.0,1.000,329.0,0.030
3천 이상,444.0,115.0,1.000,188.0,0.021


v6


,단위연도,이탈해,감지율,정상해,잘못된경보율
규모,,,,,
300~1천,771.0,280.0,0.843,285.0,0.004
1천~3천,904.0,293.0,0.997,329.0,0.027
3천 이상,444.0,115.0,1.000,188.0,0.069


## 3. 보조 정보 (결정에 쓰지 않음: 이미 본 자료)
같은 6차를 앞서 검증한 주제에 적용. 저장소의 저장본을 사용합니다.

In [5]:
rows = []
for name in ["혼인", "청년 순이동", "미분양"]:
    td = K.fetch(K.TOPICS[name], verbose=False)
    for r in ["v5", "v6"]:
        o = K.run(K.TOPICS[name], tidy=td, rule=r, verbose=False)
        rows.append(dict(주제=name, 규칙=r, 감지율=f"{o['test']['감지율']:.1%}", 잘못된경보=f"{o['test']['잘못된경보율']:.1%}",
                         구간최대_잘못된경보=f"{o['groups']['잘못된경보율'].max():.1%}"))
pd.DataFrame(rows)

,주제,규칙,감지율,잘못된경보,구간최대_잘못된경보
0,혼인,v5,71.0%,3.4%,5.1%
1,혼인,v6,77.7%,2.6%,3.4%
2,청년 순이동,v5,81.6%,7.2%,8.6%
3,청년 순이동,v6,84.8%,7.5%,10.2%
4,미분양,v5,36.3%,3.1%,6.2%
5,미분양,v6,38.9%,2.0%,3.2%


## 4. 정리
- 사전 등록 결정 규칙대로 판정합니다(위 결과). 6차가 채택되면 월간 자동 실행(`monthly_update.py`)의 규칙을 6차로 바꿉니다.
- 주의: 청년 순이동에 6차를 적용하면 한 규모 구간의 잘못된 경보가 10%를 약간 넘습니다(보조 정보). 다음 달부터 월간 리포트에서 이 구간을 따로 지켜봅니다.
- 출생은 시군구 확정치가 연 1회 공표되므로 월간 경보 대상에는 넣지 않습니다(검증용 주제).